In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROC = Path("../data/processed")
GAS = ["H2", "CH4", "C2H6", "C2H4", "C2H2"]
LABEL = "Fail"

df = pd.read_csv(PROC / "dga_clean.csv").reset_index(drop=True)

g = df[GAS].clip(lower=1.0)

print(df.shape) 

(2680, 6)


In [2]:
feat = pd.DataFrame(index=df.index)

# Log of each gas
for c in GAS:
    feat[f"log_{c}"] = np.log10(g[c])

# Overall gas level
total = g.sum(axis=1)
feat["log_total"] = np.log10(total)

# Relative composition of each gas
for c in GAS:
    feat[f"pct_{c}"] = g[c] / total

# Duval-style gas proportions
tri = g[["CH4", "C2H4", "C2H2"]]

tri_pct = tri.div(tri.sum(axis=1), axis=0)

for c in tri_pct.columns:
    feat[f"tri_{c}"] = tri_pct[c]

# Classic gas ratios
feat["r_C2H2_C2H4"] = np.log10(g["C2H2"] / g["C2H4"])
feat["r_CH4_H2"] = np.log10(g["CH4"] / g["H2"])
feat["r_C2H4_C2H6"] = np.log10(g["C2H4"] / g["C2H6"])

X = feat
y = df[LABEL]

print(X.shape)

assert not X.isna().any().any() and np.isfinite(X.values).all()

(2680, 17)


In [3]:
logkey = np.log10(g).round(2)
groups = logkey.apply(tuple, axis=1).factorize()[0]

sizes = pd.Series(groups).value_counts()
print("Unique groups:", groups.max() + 1, "of", len(df), "rows")
print("Groups with 2+ rows:", (sizes > 1).sum(), "| rows inside them:", sizes[sizes > 1].sum())
print("Largest group size:", sizes.max())

Unique groups: 2293 of 2680 rows
Groups with 2+ rows: 340 | rows inside them: 727
Largest group size: 5


In [4]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups[train_idx]

assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))
print(X_train.shape, X_test.shape)
print(y_test.value_counts())

(2144, 17) (536, 17)
Fail
NF    142
T3    114
D1     67
D2     66
T1     62
T2     44
PD     41
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "LogReg": make_pipeline(StandardScaler(),
              LogisticRegression(max_iter=3000, class_weight="balanced")),
    "RandomForest": RandomForestClassifier(
              n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
    "HistGB": HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}

cv_g = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv_g, groups=groups_train,
                         scoring=["f1_macro", "accuracy"])
    rows.append({"model": name,
                 "macro_F1": res["test_f1_macro"].mean(),
                 "macro_F1_std": res["test_f1_macro"].std(),
                 "accuracy": res["test_accuracy"].mean()})
pd.DataFrame(rows).round(3)

,model,macro_F1,macro_F1_std,accuracy
0,LogReg,0.697,0.014,0.726
1,RandomForest,0.801,0.016,0.826
2,HistGB,0.788,0.007,0.820


Evaluation	                  HistGB macro-F1
Random split (before)	      0.809
Group-aware split (after)	  0.788